In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType, TimestampType

catalog_name = "ecommerce"

In [0]:
df = spark.table("ecommerce.bronze.brnz_brands")
df.show(10)

In [0]:
df_silver = df.withColumn('brand_name',F.trim(F.col('brand_name')))
df_silver.show(5)

In [0]:
df_silver = df_silver.withColumn('brand_code', F.regexp_replace(F.col('brand_code'),r'[^A-Za-z0-9]',''))
df_silver.show(10)

In [0]:
df_silver.select("category_code").distinct().show(10)

In [0]:
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

df_silver = df_silver.replace(to_replace=anomalies, subset=["category_code"])
df_silver.select("category_code").distinct().show()

In [0]:
df_silver.show()
spark.sql("DROP TABLE IF EXISTS ecommerce.silver.slv_brands")
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable("ecommerce.silver.slv_brands")

In [0]:
df_bronze = spark.table("ecommerce.bronze.brz_category")

df_bronze.show(10)

In [0]:
df_duplicates = df_bronze.groupBy("category_code").count().filter("count > 1")
df_duplicates.show()
df_silver = df_bronze.dropDuplicates(["category_code"])
df_silver.show(10)

In [0]:
df_silver = df_silver.withColumn("category_code",F.upper(F.col("category_code")))
df_silver.show(10)

In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable("ecommerce.silver.slv_category")


In [0]:
df_bronze = spark.table("ecommerce.bronze.brz_products")
df_bronze.show(10)
row_count, columncount = df_bronze.count(), len(df_bronze.columns)
print(f"Row Count : {row_count}, Column Count : {columncount}")

In [0]:
df_bronze.select("weight_grams").show(5)

In [0]:
df_silver = df_bronze.withColumn("weight_grams",F.regexp_replace(F.col("weight_grams"),"g","").cast(IntegerType()))
df_silver.show(5)

In [0]:
df_silver.select("length_cm").show(5)
df_silver = df_silver.withColumn("length_cm",F.regexp_replace(F.col("length_cm"),",",".").cast(FloatType()))
df_silver.show(5)

In [0]:
df_silver.select("category_code","brand_code").show(2)

In [0]:
df_silver = df_silver.withColumn("category_code",F.upper(F.col("category_code")))\
             .withColumn("brand_code",F.upper(F.col("brand_code")))
df_silver.select("category_code","brand_code").show(2)



In [0]:
df_silver.show(5)
df_silver = df_silver.withColumn("material",\
            F.when(F.col("material").contains("Coton"), "Cotton")\
            .when(F.col("material").contains("Ruber"), "Rubber")\
            .when(F.col("material").contains("Alumium"), "Aluminum")\
            .otherwise(F.col("material")))
df_silver.select("material").distinct().show(5)         

In [0]:
df_silver.filter(F.col('rating_count')>0).select("rating_count").show(3)

In [0]:

# if rating_count is null, replace with 0; otherwise take the absolute value to fix negatives
df_silver = df_silver.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
     .otherwise(F.lit(0)))
df_silver.show()




In [0]:
df_silver.select(
    "weight_grams",
    "length_cm",
    "category_code",
    "brand_code",
    "material",
    "rating_count"
).show(10, truncate=False)

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("ecommerce.silver.slv_products")

In [0]:
df_bronze = spark.read.table("ecommerce.bronze.brz_customers")
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_bronze.show(10)

In [0]:
df_bronze.filter(F.col("customer_id").isNull()).show(3)
null_count = df_bronze.filter(F.col("customer_id").isNull()).count()
null_count

In [0]:
df_silver = df_bronze.dropna(subset=["customer_id"])
row_count = df_silver.count()
print(f"Row count after droping null values: {row_count}")

In [0]:
null_count = df_silver.filter(F.col("phone").isNull()).count()
null_count

In [0]:
df_silver.filter(F.col("phone").isNull()).show(3)

In [0]:
df_silver = df_silver.fillna("Not Available", subset=["phone"])

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema","true") \
    .saveAsTable("ecommerce.silver.slv_customers")

In [0]:
df_bronze = spark.read.table("ecommerce.bronze.brz_calendar")
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_bronze.show(3)

In [0]:
from pyspark.sql.functions import to_date
df_silver = df_bronze.withColumn("date", to_date(df_bronze["date"],"dd-MM-yyyy"))
print(df_silver.printSchema())
df_silver.show(5)

In [0]:
duplicates = df_silver.groupBy('date').count().filter("count > 1")
print("Total duplicated Rows: ", duplicates.count())

In [0]:
df_silver = df_silver.dropDuplicates(['date'])
print("Rows: ", df_silver.count())

In [0]:
df_silver = df_silver.withColumn("day_name", F.initcap(F.col("day_name")))

df_silver.show(5)

In [0]:
df_silver = df_silver.withColumn("week_of_year", F.abs(F.col("week_of_year")))# Convert negative to positive

df_silver.show(3)

In [0]:
df_silver = df_silver.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))))

df_silver = df_silver.withColumn("week_of_year", F.concat_ws("-", F.concat(F.lit("Week"), F.col("week_of_year"), F.lit("-"), F.col("year"))))

df_silver.show(3)

In [0]:
df_silver = df_silver.withColumnRenamed("week_of_year", "week")

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("ecommerce.silver.slv_calendar")